# **I. Data Crawling**

## **1.Sub System 2 - 제목 모음**

**'lawtext.txt'로 파일 만들어두었음**

**Selenium, BeautifulSoup 모두 활용하여 데이터 크롤링 시도하였으나, 데이터 수집 불가로 인해 직접 데이터 수집함 => 수집한 데이터는 .txt 파일로 제작해 둠**

**0. 제목 추출**

In [ ]:
import re

# 1. 파일 열어서 읽기
f = open('lawtext.txt', 'r')
text = f.read()
f.close()

# 2. < > 안에 있는 텍스트 찾기 (판결 제목)
title_list = re.findall(r'<([^<>]+)>', text)

# 3. 중복 제거 + 가나다순 정렬
final_titles = list(set(title_list))
final_titles.sort()

# 4. 출력
print("▶ 판결 제목 추출 결과:\n")

for i in range(len(final_titles)):
    print(f"{i + 1}. {final_titles[i]}")

▶ 판결 제목 추출 결과:

1. 대법원 2023. 12. 7. 선고 2023다246600 판결
2. 대법원 2023. 12. 7. 선고 2023다269139 판결


# **II. Data Preprocessing (KO-KoNLPy)**

## **0. Setting**

→ 필요한 라이브러리(KoNLPy, NLTK 등) 설치 및 기본 세팅.
→ .txt 파일 로딩 코드 작성

**KoNLPY 설치 ! (기본세팅 1)**

In [ ]:
# 라이브러리 설치하기
!apt-get update
!apt-get install g++ openjdk-8-jdk python-dev python3-dev
!pip3 install JPype1-py3
!pip3 install konlpy
# JAVA_HOME 환경변수 설정하기
%env JAVA_HOME "/usr/lib/jvm/java-8-openjdk-amd64"

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,253 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,024 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restrict

**NLTK 설치 ! (기본세팅 2)**

In [ ]:
#전부설치
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

## **1. Sub System 1 - 판결 유형 A, B 분류**

In [ ]:
import re

# 1. 텍스트 불러오기
f = open('lawtext.txt', 'r')
text = f.read()
f.close()

# 2. 판결문 나누기
split_sent = re.split(r'(<대법원\s.*?판결>)', text)

caselist = []

num = 1

while num < len(split_sent):
    title = split_sent[num]  # 홀수 번호는 제목, 짝수 번호는 내용
    if num + 1 < len(split_sent):
        content = split_sent[num + 1]
    else:
        content = ''

    eachcase = [title, content]
    caselist.append(eachcase)

    num =num + 2


# 3. 조건에 따라 분류
A_cases = []
B_cases = []

for case in caselist:
    title = case[0]
    body = case[1]
    body_lower = body.lower()

    if '원심판결' in body and '파기' in body:
        A_cases.append(title)
    elif '상고' in body and '기각' in body:
        B_cases.append(title)

# 4. 출력
print(" A 유형 (원심판결, 파기):")
for h in A_cases:
    print(h)

print("\n B 유형 (상고, 기각):")
for h in B_cases:
    print(h)

print ( '''
 A. 원심판결 파기
설명:‘원심판결 파기’는 대법원이 하급심(보통 고등법원)의 판결에 법리적 오류나 사실 판단의 잘못이 있다고 판단하여, 그 판결을 **취소(파기)**하고 다시 심리·판단하도록 **사건을 되돌려 보내는 것(환송)**을 말합니다.
즉, 하급심 판결이 틀렸다고 본 것입니다. 파기되면 해당 사건은 다시 하급심으로 내려가 새로운 재판이 진행됩니다.

예시 상황: 계약의 해석을 잘못한 경우 / 증거 채택이나 법 적용에서 오류가 있는 경우 / 사실 인정 과정에서 중대한 누락이나 왜곡이 있는 경우 등

 B. 상고 기각
설명:‘상고 기각’은 대법원이 하급심의 판결에 대해 법적으로 문제가 없다고 판단하여, 상고한 사람(패소한 당사자)의 주장을 받아들이지 않고, 하급심 판결을 그대로 확정하는 것을 의미합니다.
즉, 하급심 판결이 정당하다고 본 것입니다. 더 이상 다툴 수 없고, 판결은 최종 확정됩니다.

예시 상황: 상고 이유가 법적으로 인정되지 않는 경우 / 사실심 판단을 뒤집을 근거가 부족한 경우 / 단순한 불복이나 오해로 제기된 상고인 경우 등
''')


 A 유형 (원심판결, 파기):
<대법원 2023. 12. 7. 선고 2023다269139 판결>

 B 유형 (상고, 기각):
<대법원 2023. 12. 7. 선고 2023다246600 판결>

 A. 원심판결 파기
설명:‘원심판결 파기’는 대법원이 하급심(보통 고등법원)의 판결에 법리적 오류나 사실 판단의 잘못이 있다고 판단하여, 그 판결을 **취소(파기)**하고 다시 심리·판단하도록 **사건을 되돌려 보내는 것(환송)**을 말합니다.
즉, 하급심 판결이 틀렸다고 본 것입니다. 파기되면 해당 사건은 다시 하급심으로 내려가 새로운 재판이 진행됩니다.

예시 상황: 계약의 해석을 잘못한 경우 / 증거 채택이나 법 적용에서 오류가 있는 경우 / 사실 인정 과정에서 중대한 누락이나 왜곡이 있는 경우 등

 B. 상고 기각
설명:‘상고 기각’은 대법원이 하급심의 판결에 대해 법적으로 문제가 없다고 판단하여, 상고한 사람(패소한 당사자)의 주장을 받아들이지 않고, 하급심 판결을 그대로 확정하는 것을 의미합니다.
즉, 하급심 판결이 정당하다고 본 것입니다. 더 이상 다툴 수 없고, 판결은 최종 확정됩니다.

예시 상황: 상고 이유가 법적으로 인정되지 않는 경우 / 사실심 판단을 뒤집을 근거가 부족한 경우 / 단순한 불복이나 오해로 제기된 상고인 경우 등



# **III. Linguistic Analysis**

**언어학적 분석 step 1-1: 어떤 한국어 형태소 분석기 사용할 지 선택**

In [ ]:
# 형태소 분석기별 품사유형의 개수와 종류
from konlpy.tag import Kkma, Komoran, Hannanum, Okt
kkma = Kkma()
komoran = Komoran()
hannanum = Hannanum()
okt = Okt()

print("Kkma")
print(len(kkma.tagset), kkma.tagset, sep='\t', end='\n\n')
print("Komoran")
print(len(komoran.tagset), komoran.tagset, sep='\t', end='\n\n')
print("Hannanum")
print(len(hannanum.tagset), hannanum.tagset, sep='\t', end='\n\n')
print("Okt")
print(len(okt.tagset), okt.tagset, sep='\t')

In [ ]:
from konlpy.tag import Komoran, Kkma, Okt, Hannanum

# 분석할 예제 문장
text = "대법원은 이를 공평과 신의칙에 반하는 것으로 판단하였다."

# 분석기 객체 생성
komoran = Komoran()
kkma = Kkma()
okt = Okt()
hannanum = Hannanum()

# 형태소 분석 실행
komoran_result = komoran.pos(text)
kkma_result = kkma.pos(text)
okt_result = okt.pos(text)
hannanum_result = hannanum.pos(text)

# 결과 출력
print("===== Komoran 분석 결과 =====")
for word, tag in komoran_result:
    print(f"{word}/{tag}", end=' ')
print("\n")

print("===== Kkma 분석 결과 =====")
for word, tag in kkma_result:
    print(f"{word}/{tag}", end=' ')
print("\n")

print("===== Okt 분석 결과 =====")
for word, tag in okt_result:
    print(f"{word}/{tag}", end=' ')
print("\n")

print("===== Hannanum 분석 결과 =====")
for word, tag in hannanum_result:
    print(f"{word}/{tag}", end=' ')
print()

===== Komoran 분석 결과 =====
대법원/NNP 은/JX 이/NP 를/JKO 공평/NNG 과/JC 신의/NNP 칙/NNG 에/JKB 반하/VV 는/ETM 것/NNB 으로/JKB 판단/NNG 하/XSV 았/EP 다/EF ./SF 

===== Kkma 분석 결과 =====
대법원/NNG 은/JX 이르/VV ㄹ/ETD 공평/NNG 과/JKM 신의칙/NNG 에/JKM 반하/VV 는/ETD 것/NNB 으로/JKM 판단/NNG 하/XSV 였/EPT 다/EFN ./SF 

===== Okt 분석 결과 =====
대법원/Noun 은/Josa 이를/Verb 공평/Noun 과/Josa 신의칙/Noun 에/Josa 반하는/Adjective 것/Noun 으로/Josa 판단/Noun 하였다/Verb ./Punctuation 

===== Hannanum 분석 결과 =====
대법원/N 은/J 이/N 를/J 공평/N 과/J 신의칙/N 에/J 반하/P 는/E 것/N 으로/J 판단/N 하/X 었다/E ./S 


# FINAL !

In [ ]:
# STEP 1 - 데이터 전처리 (문장 단위 토큰화 + 기호 위주 불용어 제거)
print('-----------------------------------------------------------')
print('>>>>>>> STEP 1. 데이터 전처리 (문장 단위 토큰화 + 기호 위주 불용어 제거) <<<<<<<')

import re
from konlpy.tag import Komoran

# 1. 텍스트 파일 열기
f = open('lawtext.txt', 'r')
text = f.read()
f.close()

# 2. 판결 제목 단위로 텍스트 나누기
blocks = re.split(r'(<대법원\s.*?판결>)', text)

# 3. 제목 + 본문 묶어 사건별 리스트 생성
cases = []
for i in range(1, len(blocks), 2):  # 홀수 index: 제목, 짝수 index: 본문
    title = blocks[i].strip()
    body = blocks[i+1].strip() if i+1 < len(blocks) else ''
    full_case = f"{title}\n{body}"
    cases.append(full_case)

# 4. 전처리 함수 정의
def preprocess_sentence(sentence):
    sentence = re.sub(r'[^가-힣a-zA-Z\s]', ' ', sentence)
    sentence = re.sub(r'\b[가-하]\b', '', sentence)
    sentence = re.sub(r'\s+', ' ', sentence)
    return sentence.strip()

# 5. 사건별 전처리 및 문장 단위 토큰화
preprocessed_cases = []
print('\n==== 전처리된 결과 ====')

for case_num, case in enumerate(cases, start=1):
    print(f"\n===== 사건 {case_num} =====")

    # 6. 문장 단위 분리 (종결 표현 '다.', '요.' 기준)
    sentences = re.split(r'(?<=[다요])\.\s*', case)

    cleaned_sentences = []
    for idx, sentence in enumerate(sentences, start=1):
        clean = preprocess_sentence(sentence)
        if clean:
            print(f"{idx}. {clean}")
            cleaned_sentences.append(clean)

    preprocessed_cases.append(cleaned_sentences)

# STEP 2 - 품사 태깅 + 동사 추출
print('\n\n')
print('>>>>>>> STEP 2. 품사 태깅으로 고유 동사 목록 추출하기 <<<<<<<')

from konlpy.tag import Komoran

# 1. 형태소 분석기 객체 생성
komoran = Komoran()

# 2. 사건별 동사 추출
all_verbs = []

for case_idx, case_sentences in enumerate(preprocessed_cases, start=1):
    verbs = []
    for sentence in case_sentences:
        tagged = komoran.pos(sentence)

        for word, tag in tagged:
            if tag.startswith('VV'):
                verbs.append(word)
    all_verbs.append(verbs)

# 3. 사건별 고유 동사 목록 추출 및 정렬
unique_verbs_by_case = []
for verbs in all_verbs:
    unique_verbs = sorted(set(verbs))
    unique_verbs_by_case.append(unique_verbs)

# 4. 사건별 고유 동사 목록 출력
for idx, verbs in enumerate(unique_verbs_by_case, start=1):
    print(f"\n===== 사건 {idx}의 고유 동사 목록 ({len(verbs)}개) =====")
    print(', '.join(verbs))


# STEP 3 - PMI 기반 Bigram + Trigram Collocation 분석
print('\n\n')
print('>>>>>>> STEP 3. 전처리 문장을 기반으로 PMI Bigram/Trigram 분석 <<<<<<<')

# 1. 필요한 라이브러리 불러오기
from konlpy.tag import Komoran
from nltk.collocations import BigramCollocationFinder, TrigramCollocationFinder
from nltk.collocations import BigramAssocMeasures, TrigramAssocMeasures

# 2. 형태소 분석기 객체 생성
komoran = Komoran()

# 3. 전체 문장을 하나의 리스트로 평탄화
all_sentences = []
for case in preprocessed_cases:
    for sentence in case:
        all_sentences.append(sentence)

# 4. 모든 문장을 하나의 문자열로 합치기
joined_text = ' '.join(all_sentences)

# 5. 형태소 분석
words = komoran.morphs(joined_text)

# 6. Bigram Collocation 분석
bigram_finder = BigramCollocationFinder.from_words(words)
bigram_finder.apply_freq_filter(2)  # 최소 2회 이상 등장
bigram_measures = BigramAssocMeasures()
top_bigrams = bigram_finder.nbest(bigram_measures.pmi, 5)

# 7. Trigram Collocation 분석
trigram_finder = TrigramCollocationFinder.from_words(words)
trigram_finder.apply_freq_filter(2)  # 최소 2회 이상 등장
trigram_measures = TrigramAssocMeasures()
top_trigrams = trigram_finder.nbest(trigram_measures.pmi, 5)

# 8. 결과 출력
print('\n[PMI 기준 상위 2-gram]')
for i, bigram in enumerate(top_bigrams, start=1):
    print(f"{i}. {bigram[0]} + {bigram[1]}")

print('\n[PMI 기준 상위 3-gram]')
for i, trigram in enumerate(top_trigrams, start=1):
    print(f"{i}. {trigram[0]} + {trigram[1]} + {trigram[2]}")

# STEP 4 - 사건별 중요 문장 1개씩 추출
print('\n\n')
print('>>>>>>> STEP 4. 사건별 중요 문장 상위 1개 추출 (TF-IDF 기반) <<<<<<<')

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 사건별로 중요 문장 추출
case_num = 1
for case in preprocessed_cases:

    if len(case) == 0:
        print(f"\n사건 {case_num}: 문장 없음")
        case_num += 1
        continue
    elif len(case) == 1:
        print(f"\n사건 {case_num}:")
        print(f"1. {case[0]}")
        case_num += 1
        continue


    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(case)

    sentence_scores = X.sum(axis=1).A1

    top_idx = sentence_scores.argmax()

    # 결과 출력
    print(f"\n사건 {case_num}의 중요 문장:")
    print(f"{case[top_idx]}")
    case_num += 1


# STEP 5 - 중요 문장 추출 및 불필요한 표현 제거
print('\n\n')
print('======= STEP 5. 사건별 중요 문장 추출 및 불필요한 표현 제거 =======')

import re
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np



# 제거할 관용 표현
remove_phrases = [
    r'대법원\s*선고\s*판결',
    r'판결\s*요지',
    r'이 사건은',
    r'결론',
    r'주문과\s*같이\s*판결한다'
]


important_sentences = []

for case_num, case in enumerate(preprocessed_cases, start=1):
    print(f"\n[사건 {case_num}]")

    if len(case) == 0:
        print("문장이 없어 처리하지 않음.")
        important_sentences.append("")
        continue

    # TF-IDF 기반 가장 중요한 문장 1개 선택
    if len(case) == 1:
        important = case[0]
    else:
        vectorizer = TfidfVectorizer()
        X = vectorizer.fit_transform(case)
        sentence_scores = X.sum(axis=1).A1
        top_idx = sentence_scores.argmax()
        important = case[top_idx]

    # 불필요한 표현 제거
    for phrase in remove_phrases:
        important = re.sub(phrase, '', important)

    important_sentences.append(important)
    print(f" 중요 문장 (정제 후): {important}")



# STEP 6. 쉬운 말 해설 (용어 설명 사전 적용)
print('\n\n')
print('======= STEP 6. 쉬운 말 해설 (용어 설명 사전 적용) =======')

easy_dict = {
    "기각하": "요청을 받아들이지 않음",
    "인용하": "요청을 받아들임",
    "간주하": "그렇게 본다",
    "파기하": "이전 판결을 무효로 함",
    "환송하": "다시 판단하도록 보냄",
    "취소하": "효력을 없앰",
    "수용하": "받아들임",
    "위반하": "규칙을 어김",
    "면제하": "책임을 면함",
    "부담하": "책임이나 의무를 가짐",
    "성립하": "성립되었다고 봄",
    "효력있": "법적인 효력이 있음",
    "효력없": "법적인 효력이 없음",
    "위법하": "법을 위반함",
    "적법하": "법에 맞음",
    "정당하": "옳고 타당함",
    "정당화되": "정당하다고 인정됨",
    "추정하": "그렇다고 예상함",
    "추단하": "추론하여 판단함",
    "원고": "소송을 건 사람",
    "피고": "소송을 당한 사람",
    "판결": "재판 결과",
    "청구": "요청",
    "이행": "실제로 함",
    "계약": "약속",
    "채무": "갚아야 할 돈 또는 의무",
    "채권자": "돈을 받을 사람",
    "채무자": "돈을 갚아야 하는 사람",
    "의무": "해야 할 일",
    "권리": "누릴 수 있는 자격"
}

def add_easy_explanation(text, glossary):
    for word, meaning in glossary.items():
        pattern = r'\b' + re.escape(word) + r'\b(?!\s*\()'
        replacement = f"{word}({meaning})"
        text = re.sub(pattern, replacement, text)
    return text

print('\n[사건별 중요 문장 요약 (용어 설명 포함)]')

for case_num, sentence in enumerate(important_sentences, start=1):
    final_sentence = add_easy_explanation(sentence, easy_dict)
    print(f"\n사건 {case_num}의 중요 문장 요약:")

    split_sentences = re.split(r'(?<=[다요])\.', final_sentence)
    split_sentences = [s.strip() for s in split_sentences if s.strip() != '']

    for i, chunk in enumerate(split_sentences, start=1):
        print(f"{i}) {chunk}.")

-----------------------------------------------------------
>>>>>>> STEP 1. 데이터 전처리 (문장 단위 토큰화 + 기호 위주 불용어 제거) <<<<<<<

==== 전처리된 결과 ====

===== 사건 1 =====
1. 대법원 선고 판결 소유권이전등기 아파트 매매계약서에 인도일과 실제 명도일 약정이 별도로 있는 경우 매도인의 현실인도의무 인정 여부가 문제된 사건 판결요지 일반적으로 계약을 해석할 때에는 형식적인 문구에만 얽매여서는 되고 쌍방당사자의 진정한 의사가 무엇인가를 탐구하여야 한다
2. 계약 내용이 명확하지 않은 경우 계약서의 문언이 계약 해석의 출발점이지만 당사자들 사이에 계약서의 문언과 다른 내용으로 의사가 합치된 경우 의사에 따라 계약이 성립한 것으로 해석하여야 한다
3. 당사자 사이에 계약의 해석을 둘러싸고 이견이 있어 당사자의 의사 해석이 문제 되는 경우에는 계약의 형식과 내용 계약이 체결된 동기와 경위 계약으로 달성하려는 목적 당사자의 진정한 의사 거래 관행 등을 종합적으로 고려하여 논리와 경험의 법칙 그리고 사회일반의 상식과 거래의 통념에 따라 합리적으로 해석하여야 한다
4. 민법 항에서 정한 선이행의무를 지고 있는 당사자가 상대방의 이행이 곤란할 현저한 사유가 있는 때에 자기의 채무이행을 거절할 있는 경우 선이행채무를 지고 있는 당사자가 계약 성립 후 상대방의 신용불안이나 재산상태 악화 등과 같은 사정으로 상대방의 이행을 받을 없는 사정변경이 생기고 이로 말미암아 당초의 계약 내용에 따른 선이행의무를 이행하게 하는 것이 공평과 신의칙에 반하게 되는 경우를 가리킨다
5. 상대방의 채무가 아직 이행기에 이르지 않았지만 이행기에 이행될 것인지 여부가 현저히 불확실하게 경우에는 선이행채무를 지고 있는 당사자라도 상대방의 이행이 확실하게 때까지 선이행의무의 이행을 거절할 있다
6. 갑이 을로부터 아파트를 매수하기로 하는 계약을 체결하였고 계약 체결 무렵 아파트에 거주 중인 임차인 병이

# **IV - ii. 쉬운말 변환 사전 - 수동 제작**

표준국어대사전 참조

In [ ]:
easy_dict = {
    # 판단 관련 동사
    "기각하": "요청을 받아들이지 않음",
    "인용하": "요청을 받아들임",
    "간주하": "그렇게 본다",


    # 판결 처리
    "파기하": "이전 판결을 무효로 함",
    "환송하": "다시 판단하도록 보냄",
    "취소하": "효력을 없앰",


    # 수용·부정 계열
    "수용하": "받아들임",


    # 법적 의무와 책임
    "위반하": "규칙을 어김",
    "면제하": "책임을 면함",
    "부담하": "책임이나 의무를 가짐",

    # 법적 효과
    "성립하": "성립되었다고 봄",
    "효력있": "법적인 효력이 있음",
    "효력없": "법적인 효력이 없음",
    "위법하": "법을 위반함",
    "적법하": "법에 맞음",
    "정당하": "옳고 타당함",
    "정당화되": "정당하다고 인정됨",
    "반하": "반함",
    "미치": "영향이 미침",

    # 확신·의심
    "추정하": "그렇다고 예상함",
    "추단하": "추론하여 판단함",

    # 일반 명사
    "원고": "소송을 건 사람",
    "피고": "소송을 당한 사람",
    "판결": "재판 결과",
    "청구": "요청",
    "이행": "실제로 함",
    "계약": "약속",
    "채무": "갚아야 할 돈 또는 의무",
    "채권자": "돈을 받을 사람",
    "채무자": "돈을 갚아야 하는 사람",
    "의무": "해야 할 일",
    "권리": "누릴 수 있는 자격"
}